In [5]:
#IMPORTS

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import pandas as pd
import numpy as np
import torch.nn.functional as F
from sklearn.model_selection import train_test_split

In [6]:

#DATA IMPORT

# Load the data set
data_full = pd.read_csv("sample_data_for_project_final.csv")
metadata_full = pd.read_csv("sample_metadata_for_project_final.csv")

metadata_full.columns = metadata_full.columns.str.strip()

# Prepare X and y
y = metadata_full['sample_label'].astype(int).values
X = data_full.iloc[:, 1:].values.T

num_features = X.shape[1]

# Hold out a stratified 20% test set (same class balance in train and test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# Convert to tensors
X_train_tensor = torch.FloatTensor(X_train)
X_test_tensor = torch.FloatTensor(X_test)
y_train_tensor = torch.FloatTensor(y_train).view(-1, 1)
y_test_tensor = torch.FloatTensor(y_test).view(-1, 1)

# Calculate mean & std on TRAINING data only (prevents test-set leakage)
train_mean = X_train_tensor.mean(dim=0)
train_std = X_train_tensor.std(dim=0) + 1e-8

X_train_tensor = (X_train_tensor - train_mean) / train_std
# Apply the exact same transformation to the test set
X_test_tensor = (X_test_tensor - train_mean) / train_std

# Package into DataLoaders
train_data = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_data, batch_size=32, shuffle=True)

test_data = TensorDataset(X_test_tensor, y_test_tensor)
test_loader = DataLoader(test_data, batch_size=32, shuffle=False)

print(f"Training set: {X_train_tensor.shape[0]} samples")
print(f"Test set: {X_test_tensor.shape[0]} samples\n")


Training set: 691 samples
Test set: 173 samples



In [7]:
# Shared training loop
# reports AVERAGE loss per batch so numbers are comparable across models
def train_model(model, optimizer, criterion, loader, name, epochs=50):
    for epoch in range(epochs):
        model.train()
        current_loss = 0.0
        for batch_X, batch_y in loader:
            optimizer.zero_grad()
            predictions = model(batch_X)
            loss = criterion(predictions, batch_y)
            loss.backward()
            optimizer.step()
            current_loss += loss.item()
        if (epoch + 1) % 10 == 0:
            print(f"{name} Epoch {epoch+1}/{epochs} | Avg loss: {current_loss / len(loader):.4f}")


#ANN model
print(" Training ANN ")
class CustomBioNet(nn.Module):
    def __init__(self, input_features, hidden_neurons):
        super(CustomBioNet, self).__init__()
        self.hidden = nn.Linear(input_features, hidden_neurons)
        self.output = nn.Linear(hidden_neurons, 1)
        self.relu = nn.ReLU()
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        x = self.hidden(x)
        x = self.relu(x)
        x = self.output(x)
        x = self.sigmoid(x)
        return x

model_ann = CustomBioNet(input_features=num_features, hidden_neurons=10)
optimizer_ann = optim.Adam(model_ann.parameters(), lr=0.001)
train_model(model_ann, optimizer_ann, nn.BCELoss(), train_loader, "ANN")


# Linear Regression model: y = Wx + b
print("\nTraining Linear Regression ")
class LinearRegressionModel(nn.Module):
    def __init__(self, input_dim):
        super(LinearRegressionModel, self).__init__()
        self.linear = nn.Linear(input_dim, 1)

    def forward(self, x):
        return self.linear(x)

model_lin = LinearRegressionModel(input_dim=num_features)
optimizer_lin = optim.Adam(model_lin.parameters(), lr=0.0001)
train_model(model_lin, optimizer_lin, nn.MSELoss(), train_loader, "Linear Reg")


#Logistic Regression model
print("\nTraining Logistic Regression ")
class LogisticRegressionModel(nn.Module):
    def __init__(self, input_dim):
        super(LogisticRegressionModel, self).__init__()
        self.linear = nn.Linear(input_dim, 1)

    def forward(self, x):
        return torch.sigmoid(self.linear(x))

model_log = LogisticRegressionModel(input_dim=num_features)
optimizer_log = optim.Adam(model_log.parameters(), lr=0.001)
train_model(model_log, optimizer_log, nn.BCELoss(), train_loader, "Logistic Reg")


#PCA model (fit on the standardized TRAINING data)
print("\nRunning PCA")
X_centered = X_train_tensor - torch.mean(X_train_tensor, dim=0)

# Perform Singular Value Decomposition (SVD); torch returns V transposed (Vh)
U, S, Vh = torch.linalg.svd(X_centered, full_matrices=False)

# Project data onto top 2 principal components
top_k_components = Vh[:2, :]
X_pca = torch.mm(X_centered, top_k_components.T)
print(f"Original shape: {X_centered.shape} -> New PCA shape: {X_pca.shape}")

 Training ANN 
ANN Epoch 10/50 | Avg loss: 0.0002
ANN Epoch 20/50 | Avg loss: 0.0001
ANN Epoch 30/50 | Avg loss: 0.0000
ANN Epoch 40/50 | Avg loss: 0.0000
ANN Epoch 50/50 | Avg loss: 0.0000

Training Linear Regression 
Linear Reg Epoch 10/50 | Avg loss: 0.2787
Linear Reg Epoch 20/50 | Avg loss: 0.2625
Linear Reg Epoch 30/50 | Avg loss: 0.2841
Linear Reg Epoch 40/50 | Avg loss: 0.2139
Linear Reg Epoch 50/50 | Avg loss: 0.2136

Training Logistic Regression 
Logistic Reg Epoch 10/50 | Avg loss: 0.0001
Logistic Reg Epoch 20/50 | Avg loss: 0.0001
Logistic Reg Epoch 30/50 | Avg loss: 0.0000
Logistic Reg Epoch 40/50 | Avg loss: 0.0000
Logistic Reg Epoch 50/50 | Avg loss: 0.0000

Running PCA
Original shape: torch.Size([691, 15606]) -> New PCA shape: torch.Size([691, 2])


In [8]:
#Evaluation and Accuracy

print("\n Final Model Evaluations")

# 1 & 2. Classification Accuracy for ANN and Logistic Regression
def evaluate_classification(model, data_loader):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for batch_X, batch_y in data_loader:
            probabilities = model(batch_X)
            predictions = (probabilities >= 0.5).float()
            correct += (predictions == batch_y).sum().item()
            total += batch_y.size(0)
    return (correct / total) * 100

# Report train AND test accuracy: a big gap between them means overfitting
for model, name in [(model_ann, "ANN"), (model_log, "Logistic Regression")]:
    train_acc = evaluate_classification(model, train_loader)
    test_acc = evaluate_classification(model, test_loader)
    print(f"{name} | Train accuracy: {train_acc:.2f}% | Test accuracy: {test_acc:.2f}%")


# 3. R-Squared and MSE for Linear Regression (on the held-out test set)
def evaluate_regression(model, data_loader):
    model.eval()
    all_targets = []
    all_predictions = []

    with torch.no_grad():
        for batch_X, batch_y in data_loader:
            all_targets.append(batch_y)
            all_predictions.append(model(batch_X))

    all_targets = torch.cat(all_targets)
    all_predictions = torch.cat(all_predictions)

    ss_res = torch.sum((all_targets - all_predictions) ** 2).item()
    ss_tot = torch.sum((all_targets - all_targets.mean()) ** 2).item()
    r2_score = 1 - (ss_res / ss_tot)
    mse = ss_res / len(all_targets)

    print(f"Linear Regression Test R² Score: {r2_score:.4f}")
    print(f"Linear Regression Test Mean Squared Error: {mse:.4f}")

evaluate_regression(model_lin, test_loader)


# 4. Explained Variance for PCA
# Using the Singular Values (S) calculated during the SVD step
def evaluate_pca(S, total_samples):
    # Calculate eigenvalues from singular values
    eigenvalues = (S ** 2) / (total_samples - 1)

    # Calculate variance explained by each component
    explained_variance_ratio = eigenvalues / torch.sum(eigenvalues)

    pc1_var = explained_variance_ratio[0].item() * 100
    pc2_var = explained_variance_ratio[1].item() * 100

    print(f"PCA PC1 Explained Variance: {pc1_var:.2f}%")
    print(f"PCA PC2 Explained Variance: {pc2_var:.2f}%")
    print(f"PCA Total Variance (Top 2 PCs): {pc1_var + pc2_var:.2f}%")

evaluate_pca(S, X_train_tensor.shape[0])


 Final Model Evaluations
ANN | Train accuracy: 100.00% | Test accuracy: 99.42%
Logistic Regression | Train accuracy: 100.00% | Test accuracy: 98.84%
Linear Regression Test R² Score: -17835.2952
Linear Regression Test Mean Squared Error: 4451.7738
PCA PC1 Explained Variance: 6.75%
PCA PC2 Explained Variance: 5.01%
PCA Total Variance (Top 2 PCs): 11.76%


In [ ]:
#Describe your results
"""
ANN and Logistic Regression use Accuracy since they classify into categories of 0 or 1.
Linear Regression uses R^2 and Mean Squared Error (MSE) since it predicts continuous numbers.
PCA uses Explained Variance because it reduces dimensions rather than making predictions.


The ANN (99.42%) and Logistic Regression models(98.84%) using pytorch received high accuracy meaning they successfully
learned the biological patterns. The high scores but not 100% show that the models arent overfitted.

The Linear regression model recieved a R^2 score of -8.86. This demonstrates how linear regression is 
the wrong tool for binary classifications since the model model tried to draw a straight, continuous line 
to hit exact 0s and 1s. This highlights the necessity of activation functions like Sigmoid for classification 
tasks,as continuous linear models cannot properly bound categorical targets in high-D data. With more features(15,606)
than samples(690), the model overfits the training data and its unbounded predictions fail badly on new samples.

The PCA captured about 12% of the total variance meaning the biological signal is highly distributed across
hundreds of different variables rather than being driven by one or two.

"""